# Token and Cost Tracking: Know What Your Agent Costs

Based on: [Don't Break the Cache: Prompt Caching for Long-Horizon Tasks](https://arxiv.org/abs/2601.06007) (Jan 2026)

## The Problem

You run an agent and it works. But how many tokens did it use? How much did it cost? If you run 1,000 queries/day, can you afford it?

Strands Agents captures token usage automatically on every invocation via `result.metrics`. No extra setup needed.

## What Are Tokens?

LLMs process text in **tokens** — chunks of roughly 4 characters (a word is typically 1-2 tokens). Costs are based on token counts:

- **Input tokens:** Everything sent *to* the model — your system prompt, conversation history, tool definitions, and the user's query. You pay per input token. More tools and longer conversations mean more input tokens.
- **Output tokens:** Everything the model generates *back* — its response text, tool call arguments, and reasoning. Output tokens are typically 3-4x more expensive than input tokens because generation is more computationally intensive.
- **Cache read tokens:** When the same prompt prefix is sent repeatedly, the provider may cache it. Cached tokens cost up to 90% less than regular input tokens. This matters for agents that share the same system prompt and tool definitions across many queries.

## What We Measure

| Metric | Source | Why It Matters |
|--------|--------|---------------|
| Input tokens | `result.metrics.accumulated_usage["inputTokens"]` | Main cost driver for agents with many tools |
| Output tokens | `result.metrics.accumulated_usage["outputTokens"]` | 3-4x more expensive per token than input |
| Cache reads | `result.metrics.accumulated_usage.get("cacheReadInputTokens", 0)` | Cached tokens cost up to 90% less |
| Latency | `result.metrics.accumulated_metrics["latencyMs"]` | Affects user experience |
| Cycle count | `result.metrics.cycle_count` | Each cycle = 1 model call; more cycles = more cost |

> **What to look for:** Simple queries (weather) should use fewer tokens and cycles than complex queries (flights + weather). The cost per query tells you whether you can afford this model at scale. Compare input vs output token counts — agents with verbose responses have high output token costs.

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands import Agent, tool
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# OpenAI pricing (per 1M tokens, approximate)
PRICING = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4.1-nano": {"input": 0.10, "output": 0.40},
}

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights."""
    return f"Flights {origin}->{destination} on {date}: BA117 $450, DL1 $520"

@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city}: 18C, partly cloudy"

agent = Agent(
    model=OpenAIModel(model_id=MODEL),
    tools=[search_flights, get_weather],
    system_prompt="You are a travel assistant.",
)

QUERIES = [
    "What's the weather in Paris?",
    "Find flights from NYC to London for Friday",
    "Find flights NYC to Tokyo and check the weather there",
]

print("=" * 70)
print("TOKEN AND COST TRACKING")
print("=" * 70)

total_input = 0
total_output = 0
total_cost = 0

for query in QUERIES:
    result = agent(query)
    m = result.metrics

    input_tokens = m.accumulated_usage.get("inputTokens", 0)
    output_tokens = m.accumulated_usage.get("outputTokens", 0)
    cache_tokens = m.accumulated_usage.get("cacheReadInputTokens", 0)
    latency = m.accumulated_metrics.get("latencyMs", 0)
    cycles = m.cycle_count

    # Compute cost
    price = PRICING.get(MODEL, {"input": 3.0, "output": 15.0})
    cost = (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

    total_input += input_tokens
    total_output += output_tokens
    total_cost += cost

    print(f"\n  Query: '{query[:50]}'")
    print(f"  Input: {input_tokens:,} tokens | Output: {output_tokens:,} tokens | Cache: {cache_tokens:,}")
    print(f"  Latency: {latency:,.0f}ms | Cycles: {cycles} | Cost: ${cost:.4f}")

    # Reset messages for independent measurements
    agent.messages = []

print(f"\n📊 Total across {len(QUERIES)} queries:")
print(f"   Input tokens:  {total_input:,}")
print(f"   Output tokens: {total_output:,}")
print(f"   Total cost:    ${total_cost:.4f}")
print(f"   Avg per query: ${total_cost/len(QUERIES):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Re-collect per-query token data by re-running (tokens were printed but not stored)
# We'll reconstruct from the agent by re-running queries
query_labels = [q[:30] + '...' if len(q) > 30 else q for q in QUERIES]

# Re-run to capture per-query metrics
input_tokens_list = []
output_tokens_list = []

for query in QUERIES:
    agent.messages = []
    result = agent(query)
    m = result.metrics
    input_tokens_list.append(m.accumulated_usage.get("inputTokens", 0))
    output_tokens_list.append(m.accumulated_usage.get("outputTokens", 0))

fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor('white')

x = np.arange(len(query_labels))
width = 0.5

bars1 = ax.bar(x, input_tokens_list, width, label='Input Tokens', color='#42A5F5', edgecolor='white')
bars2 = ax.bar(x, output_tokens_list, width, bottom=input_tokens_list, label='Output Tokens', color='#FF7043', edgecolor='white')

for i, (inp, out) in enumerate(zip(input_tokens_list, output_tokens_list)):
    total = inp + out
    ax.text(i, total + 10, f'{total:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(query_labels, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Tokens', fontsize=12)
ax.set_title('Input vs Output Tokens per Query\n(Stacked bar chart)', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()